# <center>  **<span style="font-size:80px;">Machine Learning</span>** </center>

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import sys
import os

import torch

from pathlib import Path

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split, cross_validate
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, 
    confusion_matrix, classification_report
)


sys.path.append(os.path.abspath(os.path.join("..")))
from src.config import DatasetKeys, AIConstants, Paths
Paths.init_project()

In [ ]:
from main import WaterApp

from src.water2fraud.features.preprocessor import WaterPreprocessor

from src.water2fraud.models.explainability import ModelExplainer
from src.water2fraud.models.clustering import ClusterManager
from src.water2fraud.models.autoencoder import LSTMAutoencoder
from src.water2fraud.models.dataset import get_dataloader
from src.water2fraud.models.trainer import (
    train_autoencoder, 
    detect_anomalies, 
    plot_training_history, 
    plot_reconstruction
)

In [ ]:
seed = AIConstants.RANDOM_STATE
device = "cuda" if torch.cuda.is_available() else "cpu"

images_path = Path("./AMAEM")
images_path.mkdir(parents=True, exist_ok=True)

# Pipeline

## Carga y Preprocesamiento

In [ ]:
df_raw = WaterApp._load_data()
X_sequences, metadata_df, feature_names = WaterApp._phase_1_preprocessing(df_raw)

In [ ]:
X_sequences.shape

# 4004: número de muestras
# 12: seq_len = meses
# 9: features = [m3_ratio, mes,                                                                         <- AMAEM
#                ohe_domestico, ohe_comercial, ohe_no_domestico,                                        <- OneHotEncoding
#                num_vt_barrio, porcentaje_vt_barrio, ocupaciones_vt_prov, pernocatciones_vt_prov       <- Turismo
#]

In [ ]:
X_sequences[0][0] # Mes Enero

In [ ]:
X_sequences[0][7] # Mes Agosto

## Búsqueda del Número Óptimo de Clústeres (Método del Codo)

In [ ]:
#mejor_k = ClusterManager.find_optimal_clusters(X_sequences, max_clusters=6, feature_idx=0)
#AIConstants.N_CLUSTERS_DEFAULT = mejor_k

#  > Evaluando k=01 | Inercia: 6.37
#  > Evaluando k=02 | Inercia: 4.98 
#  > Evaluando k=03 | Inercia: 4.34
#  > Evaluando k=04 | Inercia: 3.99
#  > Evaluando k=05 | Inercia: 3.77
#  > Evaluando k=06 | Inercia: 3.59
#  > Evaluando k=07 | Inercia: 3.37

AIConstants.N_CLUSTERS_DEFAULT

## Clustering

In [ ]:
labels, cluster_manager = WaterApp._phase_2_clustering(X_sequences)
metadata_df['cluster'] = labels

In [ ]:
ClusterManager.plot_cluster_samples(X_sequences, labels, images_path, feature_idx=0)

## Custom Training

In [ ]:
# Entrenamos los autoencoders delegando todo a main.py
# Al pasar plot=True, las gráficas aparecerán automáticamente debajo de la celda
modelos_entrenados = WaterApp._phase_3_training(
    X_sequences, 
    metadata_df, 
    device, 
    epochs=10,#150,     # Forzamos un número alto para probar el Early Stopping
    lr=0.001,
    plot=True
)

## Inferencia y Detección

In [ ]:
# Detectamos anomalías cruzando datos con los modelos entrenados
df_resultados = WaterApp._phase_4_detection(X_sequences, metadata_df, modelos_entrenados, feature_names, device)

# Mostramos los resultados crudos
sospechosos = df_resultados[
    (df_resultados['is_ae_anomaly'] == True) & 
    (df_resultados[DatasetKeys.USO] != "COMERCIAL") &
    (df_resultados[DatasetKeys.USO] != "NO DOMESTICO")
].sort_values(by='reconstruction_error', ascending=False)

print(f"Se han detectado {len(sospechosos)} perfiles de consumo sospechosos de ser viviendas turísticas ilegales.")
display(sospechosos.head(5))

## Inspección Visual

In [ ]:
top_n = 6

# Obtenemos los sospechosos de fraude ordenados por gravedad
sospechosos = df_resultados[df_resultados['is_ae_anomaly'] == True].sort_values(by='reconstruction_error', ascending=False)    
if not sospechosos.empty:       
    for i in range(min(top_n, len(sospechosos))):
        idx_seq = sospechosos.index[i]
        cluster_id = sospechosos.loc[idx_seq, 'cluster']

        print(f"\nTop {i+1} | Índice: {idx_seq} | Clúster: {cluster_id} | Error MAE: {sospechosos.loc[idx_seq, 'reconstruction_error']:.4f}")
        secuencia_anomala = X_sequences[idx_seq]
        modelo_evaluador = modelos_entrenados[f"ae_cluster_{cluster_id}"]
        plot_reconstruction(model=modelo_evaluador, sequence=secuencia_anomala, feature_idx=0, feature_name="Ratio Consumo", device=device)

## Explicabilidad

In [ ]:
feature_names

In [ ]:
indices_anomalos = sospechosos.index

# Extraemos del tensor 3D original solo los que dieron alerta roja
# OJO: X_sequences debe ser un tensor de PyTorch
X_anomalies_tensor = torch.tensor(X_sequences[indices_anomalos], dtype=torch.float32)

# Como "fondo de normalidad", pasamos todo el tensor (o los que dieron False)
X_normal_tensor = torch.tensor(X_sequences, dtype=torch.float32)

In [ ]:
# Lanzamos el Explicador
ModelExplainer.plot_feature_importance(
    model=modelos_entrenados[f"ae_cluster_{0}"],
    
    X_train_tensor=X_normal_tensor,
    X_anomalies_tensor=X_anomalies_tensor,
    feature_names=feature_names,
    device="cuda"
)

In [ ]:
# Lanzamos el Explicador
ModelExplainer.plot_feature_importance(
    model=modelos_entrenados[f"ae_cluster_{1}"],
    X_train_tensor=X_normal_tensor,
    X_anomalies_tensor=X_anomalies_tensor,
    feature_names=feature_names,
    device="cuda"
)